# Medical Insurance Cost Prediction Using Machine Learning
### A Two-Dataset Comparative Study  |  Predictive Analytics CA3

**Objective:** build regression models that predict medical insurance cost from personal, demographic, lifestyle and health attributes, find which factors are associated with cost, and check how the approach behaves on a **second, distinct dataset**.

---
### What YOU must do before running (read this first)

| Step | What | Where |
|---|---|---|
| 1 | Run all cells top to bottom (`Runtime -> Run all`). | Colab |
| 2 | **Dataset 1** (`insurance.csv`) is downloaded automatically from a public GitHub repository. If the download fails, the cell will ask you to **upload `insurance.csv`**. | Section 3.1 |
| 3 | **Dataset 2** (`Medicalpremium.csv`) is hosted on Kaggle, which needs a login, so it is **not** auto-downloaded. **Download it from Kaggle and upload it when the cell asks** (or drag it into the Colab Files sidebar before running). | Section 3.2 |

**Dataset sources**

* Dataset 1 - *Medical Cost Personal Datasets* (Kaggle, from Brett Lantz, *Machine Learning with R*): https://www.kaggle.com/datasets/mirichoi0218/insurance
  Direct CSV used by this notebook: https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv  (repo: https://github.com/stedy/Machine-Learning-with-R-datasets)
* Dataset 2 - *Medical Insurance Premium Prediction* (Kaggle, author `tejashvi14`, file `Medicalpremium.csv`): https://www.kaggle.com/datasets/tejashvi14/medical-insurance-premium-prediction

> **Important:** this notebook contains **no pre-written results**. Every number and chart appears only after you run the cells. Interpretation cells tell you what to read from your own output.

## 1. Business / Problem Understanding  (KDD stage 1)

Insurers must estimate how much a person is likely to cost so that pricing is fair and sustainable. People also want to understand which of their characteristics are associated with higher cost.

**Why machine learning?** Cost depends on several attributes at once (age, smoking, BMI ...), and some relationships are not straight lines. Regression models can learn such patterns from historical records.

## 2. Problem Statement and Objectives  (KDD stage 2)

**Problem statement:** *Given a person's recorded attributes, predict a continuous cost value (a regression problem).*

**Objectives**
1. Build and compare Linear Regression, Ridge, Decision Tree, Random Forest and Gradient Boosting models on Dataset 1.
2. Evaluate with MAE, RMSE and R-squared on a held-out test set, and check for overfitting.
3. Identify which features are most associated with predicted cost (permutation importance).
4. Examine a **second distinct dataset**: decide honestly whether the Dataset 1 model can be validated on it, and if not, run a separate, clearly defined experiment and compare findings cautiously.

### 2.1 Setup - imports and global settings
No extra installation is needed: Google Colab already provides pandas, NumPy, Matplotlib, Seaborn and scikit-learn.
`RANDOM_STATE` is fixed so that your results are reproducible.

In [ ]:
import os, io, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_validate, GridSearchCV, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
import sklearn

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

print("pandas", pd.__version__, "| numpy", np.__version__, "| scikit-learn", sklearn.__version__)

# ---- Save every chart as a PNG automatically (used for the GitHub README) ----
# plt.show() is wrapped: before a chart is displayed it is also saved into the "figures" folder.
# At the very end of the notebook (Section 17) all charts are zipped for download.
import re
os.makedirs("figures", exist_ok=True)
_fig_count = {"n": 0}
_original_show = plt.show

def _save_then_show(*args, **kwargs):
    for num in plt.get_fignums():
        fig = plt.figure(num)
        title = fig.axes[0].get_title() if fig.axes else "figure"
        if title.startswith(("Actual vs Predicted", "Permutation Importance")):
            title = title.split(" - ")[0]                   # file name must not depend on which model wins
        slug = re.sub(r"[^a-z0-9]+", "_", title.lower()).strip("_")[:45] or "figure"
        _fig_count["n"] += 1
        fig.savefig(f"figures/{_fig_count['n']:02d}_{slug}.png", dpi=150, bbox_inches="tight")
    _original_show(*args, **kwargs)
    plt.close("all")                                          # so each chart is saved only once

plt.show = _save_then_show

## 3. Data Collection and Source Documentation  (KDD stage 3)

The helper below loads a CSV in this order: (1) a file already in the Colab working folder, (2) a download from a documented URL (only if one is given), (3) an **upload prompt**. It never pretends a download worked: if a step fails it prints the error and moves to the next option.

In [ ]:
def load_dataset(filename, url=None):
    # 1) local file (already uploaded to Colab's Files sidebar, or downloaded earlier)
    if os.path.exists(filename):
        print(f"[OK] Using local file: {filename}")
        return pd.read_csv(filename)

    # 2) documented public URL
    if url:
        try:
            df = pd.read_csv(url)
            df.to_csv(filename, index=False)
            print(f"[OK] Downloaded from: {url}")
            return df
        except Exception as e:
            print("[!] Automatic download failed:", repr(e)[:200])

    # 3) manual upload (Colab only)
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(f"Could not find {filename}. Place it in the working folder and re-run.")
    print(f">>> ACTION REQUIRED: choose the file '{filename}' from your computer in the upload box below.")
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("No file was uploaded.")
    key = filename if filename in uploaded else next(iter(uploaded))
    if key != filename:
        print(f"[note] You uploaded '{key}', expected '{filename}'. Using it anyway.")
    with open(filename, "wb") as f:
        f.write(uploaded[key])
    print(f"[OK] Loaded uploaded file as {filename}")
    return pd.read_csv(filename)

### 3.1 Dataset 1 - Medical Cost Personal Dataset (`insurance.csv`)
* **Source:** Kaggle page https://www.kaggle.com/datasets/mirichoi0218/insurance - the Kaggle description points to the GitHub repository https://github.com/stedy/Machine-Learning-with-R-datasets, from which the CSV below is downloaded.
* **Target:** `charges` - individual medical costs billed by health insurance (USD).
* **Features:** `age`, `sex`, `bmi`, `children`, `smoker`, `region`.
* **Note on origin:** it is a widely used teaching dataset from a textbook; some documentation (e.g. the R `liver` package help page) describes it as simulated from US Census demographics. Treat it as a learning dataset, not as real insurer data.

In [ ]:
D1_URL = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv"
df1_raw = load_dataset("insurance.csv", url=D1_URL)

# Check the file really is the dataset we expect
expected_d1 = {"age", "sex", "bmi", "children", "smoker", "region", "charges"}
missing_cols = expected_d1 - set(df1_raw.columns)
assert not missing_cols, f"Dataset 1 is missing columns: {missing_cols}"
print("Dataset 1 loaded with the expected columns.")

### 3.2 Dataset 2 - Medical Insurance Premium Prediction (`Medicalpremium.csv`)
* **Source:** Kaggle, https://www.kaggle.com/datasets/tejashvi14/medical-insurance-premium-prediction (login needed to download).
* **Target:** `PremiumPrice` - yearly medical insurance **premium** in Indian rupees (INR).
* **Features:** `Age`, `Diabetes`, `BloodPressureProblems`, `AnyTransplants`, `AnyChronicDiseases`, `Height`, `Weight`, `KnownAllergies`, `HistoryOfCancerInFamily`, `NumberOfMajorSurgeries`.

> **ACTION REQUIRED:** download `Medicalpremium.csv` from the Kaggle link above and upload it when the next cell shows the upload box (or put it in the Colab Files sidebar first). No download URL is used here because Kaggle needs authentication and I have not invented one.

In [ ]:
df2_raw = load_dataset("Medicalpremium.csv", url=None)

expected_d2 = {"Age", "Diabetes", "BloodPressureProblems", "AnyTransplants", "AnyChronicDiseases",
               "Height", "Weight", "KnownAllergies", "HistoryOfCancerInFamily",
               "NumberOfMajorSurgeries", "PremiumPrice"}
missing_cols = expected_d2 - set(df2_raw.columns)
assert not missing_cols, f"Dataset 2 is missing columns: {missing_cols}. Did you upload the right file?"
print("Dataset 2 loaded with the expected columns. Rows:", len(df2_raw),
      "(the Kaggle listing describes 986 rows - check that yours matches)")

## 4. Data Understanding and Initial Inspection  (KDD stage 4)
For each dataset we display shape, columns, data types, target variable, unique counts, missing values and duplicate rows.

In [ ]:
def inspect(df, name, target):
    print("=" * 78)
    print(f"{name}")
    print("=" * 78)
    print("Rows x columns :", df.shape)
    print("Target variable:", target)
    info = pd.DataFrame({"dtype": df.dtypes.astype(str),
                         "missing values": df.isna().sum(),
                         "unique values": df.nunique()})
    display(info)
    print("Exact duplicate rows:", int(df.duplicated().sum()))
    print("\nFirst 5 rows:")
    display(df.head())
    print("Summary statistics (numeric columns):")
    display(df.describe().T)

inspect(df1_raw, "DATASET 1 - insurance.csv (Medical Cost Personal Dataset)", "charges")

In [ ]:
inspect(df2_raw, "DATASET 2 - Medicalpremium.csv (Medical Insurance Premium Prediction)", "PremiumPrice")

### 4.1 Are the two datasets compatible?
Before any modelling we compare what the two targets *mean*. Do not merge datasets just because a column name looks alike.

| Aspect | Dataset 1 | Dataset 2 |
|---|---|---|
| Target | `charges` - medical costs **billed by health insurance** | `PremiumPrice` - yearly insurance **premium** |
| Currency | US dollars (USD) | Indian rupees (INR) |
| Concept | cost of care billed | price charged to the customer for cover |
| Population | US (4 US regions) | not documented in the file (INR suggests India) |
| Predictors | demographics + lifestyle (age, sex, BMI, children, smoker, region) | age + **health history flags** (diabetes, BP problems, transplants, chronic disease, allergies, family cancer history, surgeries) + height/weight |

The code below checks the overlap of columns and the target scales instead of assuming anything.

In [ ]:
cols1 = {c.lower() for c in df1_raw.columns}
cols2 = {c.lower() for c in df2_raw.columns}
shared = sorted(cols1 & cols2)
print("Column names shared by both datasets (case-insensitive):", shared)
print("Features only in Dataset 1:", sorted(cols1 - cols2 - {"charges"}))
print("Features only in Dataset 2:", sorted(cols2 - cols1 - {"premiumprice"}))

scale = pd.DataFrame({
    "Dataset 1 charges (USD)": df1_raw["charges"].describe(),
    "Dataset 2 PremiumPrice (INR)": df2_raw["PremiumPrice"].describe(),
})
display(scale)
print("Rows in Dataset 1:", len(df1_raw), "| Rows in Dataset 2:", len(df2_raw))

**Compatibility decision (based on the definitions above):**

* The targets are **different concepts in different currencies** (billed medical cost in USD vs. yearly premium in INR). A model trained on Dataset 1 cannot be scored on Dataset 2's target in any meaningful way - an error in "dollars" versus "rupees" would be nonsense.
* The datasets share essentially **only age** (BMI can be derived in Dataset 2 from height and weight). Dataset 1's strongest-looking predictors (`smoker`, `region`, `children`, `sex`) do not exist in Dataset 2, and Dataset 2's health-history flags do not exist in Dataset 1.
* Therefore **direct external validation is not valid**, and we will **not force the Dataset 1 model onto Dataset 2**.

**Plan (the defensible alternative):**
1. **Experiment A:** full model development on Dataset 1 (target `charges`), internal train/test evaluation.
2. **Experiment B:** an independent experiment on Dataset 2 (target `PremiumPrice`) with the *same methodology* (same models, same metrics, CV, tuning, permutation importance).
3. Compare the two experiments **cautiously** using unit-free measures only (R-squared and error relative to the average target), and discuss differences in data source, features, population and target definition.

## 5. Data Quality Assessment  (KDD stage 5)
Checks: missing values, duplicates, invalid category labels, impossible values.

In [ ]:
def quality_report(df, name, cat_cols=()):
    print(f"--- {name} ---")
    print("Total missing cells:", int(df.isna().sum().sum()))
    print("Duplicate rows     :", int(df.duplicated().sum()))
    for c in cat_cols:
        print(f"Levels of '{c}':", sorted(df[c].astype(str).unique()))
    print()

quality_report(df1_raw, "Dataset 1", cat_cols=["sex", "smoker", "region"])
quality_report(df2_raw, "Dataset 2")

# Sanity ranges
print("Dataset 1 ranges  -> age:", df1_raw["age"].min(), "-", df1_raw["age"].max(),
      "| bmi:", df1_raw["bmi"].min(), "-", df1_raw["bmi"].max(),
      "| children:", df1_raw["children"].min(), "-", df1_raw["children"].max(),
      "| charges min:", df1_raw["charges"].min())
print("Dataset 2 ranges  -> Age:", df2_raw["Age"].min(), "-", df2_raw["Age"].max(),
      "| Height:", df2_raw["Height"].min(), "-", df2_raw["Height"].max(),
      "| Weight:", df2_raw["Weight"].min(), "-", df2_raw["Weight"].max(),
      "| PremiumPrice min:", df2_raw["PremiumPrice"].min())

## 6. Data Cleaning and Preprocessing  (KDD stage 6)

**Decisions (documented):**
1. **Duplicates** are removed *before* splitting. Otherwise the same record could land in both the training and the test set and make the test score look better than it is.
2. **Missing values:** if any are found, the code below stops and tells you. (Imputation, if ever needed, would go **inside the scikit-learn Pipeline** so it is fitted on training data only.)
3. Text categories are lower-cased and stripped of spaces to avoid "Yes" / "yes " mismatches.
4. **Outliers are kept** (see EDA section): high charges are real, plausible values for smokers, and tree-based models are robust to them. Removing them would hide exactly the cases an insurer cares about.
5. **Encoding and scaling are NOT done here.** They are done inside a `ColumnTransformer` + `Pipeline` that is fitted on the training split only, which prevents data leakage.

In [ ]:
# ---- Dataset 1
df1 = df1_raw.copy()
for c in ["sex", "smoker", "region"]:
    df1[c] = df1[c].astype(str).str.strip().str.lower()
n_dup1 = int(df1.duplicated().sum())
df1 = df1.drop_duplicates().reset_index(drop=True)
assert df1.isna().sum().sum() == 0, "Dataset 1 has missing values - add imputation to the pipeline."
print(f"Dataset 1: removed {n_dup1} duplicate row(s). Rows now: {len(df1)}")

# ---- Dataset 2
df2 = df2_raw.copy()
n_dup2 = int(df2.duplicated().sum())
df2 = df2.drop_duplicates().reset_index(drop=True)
assert df2.isna().sum().sum() == 0, "Dataset 2 has missing values - add imputation to the pipeline."
binary_cols_d2 = ["Diabetes", "BloodPressureProblems", "AnyTransplants",
                  "AnyChronicDiseases", "KnownAllergies", "HistoryOfCancerInFamily"]
for c in binary_cols_d2:
    assert set(df2[c].unique()) <= {0, 1}, f"{c} is not 0/1 coded"
# Derived feature (row-by-row calculation, uses no other rows and not the target => no leakage)
df2["BMI"] = df2["Weight"] / (df2["Height"] / 100) ** 2
print(f"Dataset 2: removed {n_dup2} duplicate row(s). Rows now: {len(df2)}")
print("Dataset 2 derived BMI range:", round(df2['BMI'].min(), 1), "-", round(df2['BMI'].max(), 1))

## 7. Exploratory Data Analysis - Dataset 1  (KDD stage 7)

EDA means *looking at the data before modelling*. Each chart below has a title and labelled axes, and the code prints the key numbers so you can write your interpretation from **your own output**. Under every chart, read the printed numbers and compare them with the chart.

### 7.1 Target variable: distribution of `charges`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.histplot(df1["charges"], bins=40, kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Distribution of Insurance Charges (Dataset 1)")
axes[0].set_xlabel("Charges (USD)"); axes[0].set_ylabel("Number of people")

sns.histplot(np.log1p(df1["charges"]), bins=40, kde=True, ax=axes[1], color="darkorange")
axes[1].set_title("Distribution of log(1 + Charges)")
axes[1].set_xlabel("log(1 + Charges)"); axes[1].set_ylabel("Number of people")
plt.tight_layout(); plt.show()

print(df1["charges"].describe().round(2))
print(f"\nSkewness of charges: {df1['charges'].skew():.2f}  |  skewness after log transform: {np.log1p(df1['charges']).skew():.2f}")
print(f"Mean charges {df1['charges'].mean():,.0f} vs median {df1['charges'].median():,.0f}")

**How to read this:** a positive skewness and a mean above the median mean a long right tail - a minority of people have much higher charges than the rest. This matters for modelling: errors on the expensive cases will dominate RMSE. Write one sentence on what *your* printed skewness and mean-vs-median show.

### 7.2 Age and BMI distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.histplot(df1["age"], bins=25, kde=False, ax=axes[0], color="seagreen")
axes[0].set_title("Age Distribution (Dataset 1)")
axes[0].set_xlabel("Age (years)"); axes[0].set_ylabel("Number of people")

sns.histplot(df1["bmi"], bins=30, kde=True, ax=axes[1], color="mediumpurple")
for x, lab in [(18.5, "18.5 underweight limit"), (25, "25 overweight"), (30, "30 obese")]:
    axes[1].axvline(x, linestyle="--", color="gray")
axes[1].set_title("BMI Distribution (dashed lines at 18.5, 25, 30)")
axes[1].set_xlabel("BMI (kg/m^2)"); axes[1].set_ylabel("Number of people")
plt.tight_layout(); plt.show()

print(f"Age: min {df1['age'].min()}, median {df1['age'].median()}, max {df1['age'].max()}")
print(f"BMI: mean {df1['bmi'].mean():.1f}, median {df1['bmi'].median():.1f}")
print(f"Share of people with BMI >= 30: {(df1['bmi'] >= 30).mean():.1%}")
print(f"Share of smokers: {(df1['smoker'] == 'yes').mean():.1%}")

**How to read this:** check whether ages are spread fairly evenly or concentrated, and what share of the sample has BMI of 30 or more (the usual cut-off for obesity). Note these are *sample* shares, not statements about a whole population.

### 7.3 Charges by smoking status

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
sns.boxplot(data=df1, x="smoker", y="charges", order=["no", "yes"], ax=axes[0], palette="Set2")
axes[0].set_title("Charges by Smoking Status")
axes[0].set_xlabel("Smoker"); axes[0].set_ylabel("Charges (USD)")

sns.scatterplot(data=df1, x="age", y="charges", hue="smoker", alpha=0.6, ax=axes[1])
axes[1].set_title("Charges vs Age, coloured by Smoking Status")
axes[1].set_xlabel("Age (years)"); axes[1].set_ylabel("Charges (USD)")
plt.tight_layout(); plt.show()

g = df1.groupby("smoker")["charges"].agg(["count", "mean", "median"]).round(0)
display(g)
ratio = g.loc["yes", "median"] / g.loc["no", "median"]
print(f"Median charges of smokers are {ratio:.1f} times the median of non-smokers in this sample.")

**How to read this:** compare the two medians in the table and the position of the two colour groups in the scatter plot. Look at whether the age-charges pattern forms separate bands for the colours. This is an *association in this dataset*, not proof of cause.

### 7.4 Charges against BMI and other variables

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
sns.scatterplot(data=df1, x="bmi", y="charges", hue="smoker", alpha=0.6, ax=axes[0])
axes[0].axvline(30, linestyle="--", color="gray")
axes[0].set_title("Charges vs BMI, coloured by Smoking Status")
axes[0].set_xlabel("BMI (kg/m^2)"); axes[0].set_ylabel("Charges (USD)")

sns.barplot(data=df1, x="children", y="charges", estimator=np.mean, errorbar=None, ax=axes[1], color="slateblue")
axes[1].set_title("Average Charges by Number of Children")
axes[1].set_xlabel("Number of children"); axes[1].set_ylabel("Average charges (USD)")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.barplot(data=df1, x="sex", y="charges", estimator=np.mean, errorbar=None, ax=axes[0], color="teal")
axes[0].set_title("Average Charges by Sex"); axes[0].set_xlabel("Sex"); axes[0].set_ylabel("Average charges (USD)")
sns.barplot(data=df1, x="region", y="charges", estimator=np.mean, errorbar=None, ax=axes[1], color="goldenrod")
axes[1].set_title("Average Charges by Region"); axes[1].set_xlabel("Region"); axes[1].set_ylabel("Average charges (USD)")
plt.tight_layout(); plt.show()

print("Average charges by sex:\n", df1.groupby("sex")["charges"].mean().round(0))
print("\nAverage charges by region:\n", df1.groupby("region")["charges"].mean().round(0))
print("\nAverage charges by children:\n", df1.groupby("children")["charges"].mean().round(0))

**How to read this:** in the BMI scatter, see whether high-BMI points are only expensive for one colour group (an *interaction*). Compare the size of differences between sexes and regions with the difference between smokers and non-smokers; small differences suggest weak associations. We test this idea properly in feature engineering and permutation importance.

### 7.5 Correlation heatmap

In [ ]:
corr_df = df1[["age", "bmi", "children", "charges"]].copy()
corr_df["smoker_yes"] = (df1["smoker"] == "yes").astype(int)   # 0/1 version for correlation only
corr = corr_df.corr()

plt.figure(figsize=(6.5, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Correlation Heatmap (Dataset 1, numeric + smoker as 0/1)")
plt.tight_layout(); plt.show()

print("Correlation with charges (sorted):")
print(corr["charges"].drop("charges").sort_values(ascending=False).round(3))

**How to read this:** Pearson correlation only measures *straight-line* association. A feature can matter a lot and still have a modest correlation (for example if its effect only appears together with another feature). Compare the sorted list with the charts above.

### 7.6 Outlier analysis (IQR rule)
A value is flagged if it lies more than 1.5 x IQR below Q1 or above Q3. Flagged does not mean wrong.

In [ ]:
def iqr_outliers(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return int(((s < lo) | (s > hi)).sum()), lo, hi

rows = []
for c in ["age", "bmi", "children", "charges"]:
    n, lo, hi = iqr_outliers(df1[c])
    rows.append({"feature": c, "outliers flagged": n, "share": n / len(df1), "lower fence": lo, "upper fence": hi})
display(pd.DataFrame(rows).set_index("feature"))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.3))
sns.boxplot(y=df1["bmi"], ax=axes[0], color="mediumpurple")
axes[0].set_title("BMI - Outlier Check"); axes[0].set_ylabel("BMI (kg/m^2)")
sns.boxplot(y=df1["charges"], ax=axes[1], color="steelblue")
axes[1].set_title("Charges - Outlier Check"); axes[1].set_ylabel("Charges (USD)")
sns.boxplot(data=df1, x="smoker", y="charges", order=["no", "yes"], ax=axes[2], palette="Set2")
axes[2].set_title("Charges - Outlier Check within Smoking Groups")
axes[2].set_xlabel("Smoker"); axes[2].set_ylabel("Charges (USD)")
plt.tight_layout(); plt.show()

ns_out = iqr_outliers(df1.loc[df1["smoker"] == "no", "charges"])[0]
sm_out = iqr_outliers(df1.loc[df1["smoker"] == "yes", "charges"])[0]
print(f"Charges outliers inside non-smokers: {ns_out} | inside smokers: {sm_out}")

**How to read this:** many "outliers" in overall `charges` may disappear when you look inside each smoking group - they are the smoker group, not errors. **Decision:** keep them (documented in Section 6).

## 8. Feature Engineering and Encoding  (KDD stage 8)

* **Categorical variables** (`sex`, `smoker`, `region`) are **one-hot encoded** inside the pipeline (each category becomes a 0/1 column). Models cannot use text directly.
* **Numeric variables** (`age`, `bmi`, `children`) are **standardised** (mean 0, std 1). This matters for Linear/Ridge regression; trees do not need it but it does no harm. The scaler learns its mean/std from the **training data only**.
* **Engineered features** (tested as an *option*, chosen by cross-validation on training data, not by guessing):
  * `obese` = 1 if BMI >= 30
  * `smoker_obese` = 1 if the person is a smoker **and** obese (an interaction, because EDA suggested BMI may matter differently for smokers)

  These are calculated row by row from that row's own inputs - they do **not** use the target or other rows, so they cannot leak information.

In [ ]:
def add_features(df):
    out = df.copy()
    out["obese"] = (out["bmi"] >= 30).astype(int)
    out["smoker_obese"] = ((out["smoker"] == "yes") & (out["bmi"] >= 30)).astype(int)
    return out

df1_fe = add_features(df1)

# Visual check of the idea behind the interaction feature
plt.figure(figsize=(7.5, 4.8))
sns.boxplot(data=df1_fe, x="smoker", y="charges", hue="obese", order=["no", "yes"], palette="Set1")
plt.title("Charges by Smoking Status and Obesity (BMI >= 30)")
plt.xlabel("Smoker"); plt.ylabel("Charges (USD)")
plt.legend(title="Obese (1 = yes)")
plt.tight_layout(); plt.show()

print(df1_fe.groupby(["smoker", "obese"])["charges"].agg(["count", "median"]).round(0))

**How to read this:** if the median for *smoker + obese* is much higher than for *smoker + not obese*, while for non-smokers obesity makes little difference, that supports adding the interaction feature. Whether it actually improves predictions is decided by cross-validation in Section 10.

## 9. Train-Test Split Without Data Leakage  (KDD stage 9)

* 80 % training / 20 % testing, **one split used for every model** so comparisons are fair.
* Stratified by `smoker` so both parts contain a similar share of smokers.
* The **test set is not used** for fitting, tuning or choosing the model. Selection is based on **5-fold cross-validation inside the training set**; the test set is used only to report the final score.
* The target `charges` is separated from the features (`X`), so it can never be used as an input.

In [ ]:
X = df1_fe.drop(columns=["charges"])
y = df1_fe["charges"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=df1_fe["smoker"])

print("Training rows:", len(X_train), "| Test rows:", len(X_test))
print("Smoker share  train:", round((X_train['smoker'] == 'yes').mean(), 3),
      "| test:", round((X_test['smoker'] == 'yes').mean(), 3))
assert "charges" not in X_train.columns, "Target leaked into the features!"

## 10. Model Selection and Development  (KDD stage 10)

**Models compared** (same split, same preprocessing):

| Model | Idea in one line |
|---|---|
| Mean baseline (Dummy) | always predicts the average training charge - the "do-nothing" yardstick |
| Linear Regression | fits one straight-line formula to all features |
| Ridge Regression | linear regression with a penalty that keeps coefficients small |
| Decision Tree | a flowchart of yes/no questions |
| Random Forest | many trees on random subsamples, predictions averaged |
| Gradient Boosting | trees built one after another, each correcting the previous errors |

We do **not** assume a winner; we let cross-validation decide.

### 10.1 Reusable pipeline helpers
A `Pipeline` bundles *preprocessing + model*, so when we call `fit`, scaling and encoding are learned from training data only, and `predict` applies them to unseen data safely.

In [ ]:
CV = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def regression_metrics(y_true, y_pred):
    return {"MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
            "R2": r2_score(y_true, y_pred)}

def build_pipeline(preprocessor, model):
    return Pipeline([("prep", preprocessor), ("model", model)])

def make_prep_d1(feature_set):
    num = ["age", "bmi", "children"]
    cat = ["sex", "smoker", "region"]
    parts = [("num", StandardScaler(), num),
             ("cat", OneHotEncoder(handle_unknown="ignore"), cat)]
    if feature_set == "Engineered":
        parts.append(("eng", "passthrough", ["obese", "smoker_obese"]))
    return ColumnTransformer(parts, remainder="drop")

def untuned_models():
    return {"Linear Regression": LinearRegression(),
            "Ridge Regression": Ridge(alpha=1.0),
            "Decision Tree": DecisionTreeRegressor(random_state=RANDOM_STATE),
            "Random Forest": RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE),
            "Gradient Boosting": GradientBoostingRegressor(random_state=RANDOM_STATE)}

### 10.2 Does the engineered feature set help? (decided by cross-validation on training data only)

In [ ]:
rows = []
for fs in ["Base", "Engineered"]:
    for name, est in untuned_models().items():
        pipe = build_pipeline(make_prep_d1(fs), est)
        cv_res = cross_validate(pipe, X_train, y_train, cv=CV,
                                scoring={"rmse": "neg_root_mean_squared_error", "r2": "r2"})
        rows.append({"Feature set": fs, "Model": name,
                     "CV RMSE": -cv_res["test_rmse"].mean(), "CV R2": cv_res["test_r2"].mean()})
fs_table = pd.DataFrame(rows)
display(fs_table.pivot(index="Model", columns="Feature set", values="CV RMSE").round(1))

avg_rmse = fs_table.groupby("Feature set")["CV RMSE"].mean()
BEST_FS = avg_rmse.idxmin()
print("Average CV RMSE across models:\n", avg_rmse.round(1))
print(f"\n==> Feature set chosen for the rest of Experiment A: '{BEST_FS}' (lower average CV RMSE on training data)")

**How to read this:** the rule is simple and fixed in advance - choose the feature set with the lower average cross-validated RMSE. Look at the table to see *which models* benefit (linear models often gain most from an interaction feature; trees can learn interactions on their own).

## 11. Hyperparameter Tuning  (KDD stage 11)

*Hyperparameters* are settings chosen **before** training (e.g. tree depth). We try a small grid for each tree-based model with 5-fold cross-validation on the **training set**, scoring by RMSE, and keep the best combination. Linear and Ridge are left at defaults (Ridge's penalty is fixed at 1.0 to keep the notebook simple; a `RidgeCV` search is listed as a future improvement).

> The grids are small so the notebook finishes in a few minutes on Colab.

In [ ]:
GRIDS = {
    "Decision Tree": {"model__max_depth": [3, 4, 5, 6, 8, None],
                      "model__min_samples_leaf": [1, 5, 10, 20]},
    "Random Forest": {"model__n_estimators": [200, 400],
                      "model__max_depth": [4, 6, 8, None],
                      "model__min_samples_leaf": [1, 3, 5]},
    "Gradient Boosting": {"model__n_estimators": [100, 200, 300],
                          "model__learning_rate": [0.03, 0.05, 0.1],
                          "model__max_depth": [2, 3, 4]},
}

def tune_and_select(preproc_fn, X_tr, y_tr, X_te, y_te, grids, label):
    candidates = {"Mean baseline (Dummy)": (DummyRegressor(), None),
                  "Linear Regression": (LinearRegression(), None),
                  "Ridge Regression": (Ridge(alpha=1.0), None),
                  "Decision Tree": (DecisionTreeRegressor(random_state=RANDOM_STATE), grids["Decision Tree"]),
                  "Random Forest": (RandomForestRegressor(random_state=RANDOM_STATE), grids["Random Forest"]),
                  "Gradient Boosting": (GradientBoostingRegressor(random_state=RANDOM_STATE), grids["Gradient Boosting"])}
    rows, fitted, best_params = [], {}, {}
    for name, (est, grid) in candidates.items():
        pipe = build_pipeline(preproc_fn(), est)
        if grid:
            gs = GridSearchCV(pipe, grid, cv=CV, scoring="neg_root_mean_squared_error", n_jobs=-1)
            gs.fit(X_tr, y_tr)
            model = gs.best_estimator_
            best_params[name] = {k.replace("model__", ""): v for k, v in gs.best_params_.items()}
        else:
            model = pipe.fit(X_tr, y_tr)
        cv_res = cross_validate(model, X_tr, y_tr, cv=CV,
                                scoring={"rmse": "neg_root_mean_squared_error", "r2": "r2"})
        tr = regression_metrics(y_tr, model.predict(X_tr))
        te = regression_metrics(y_te, model.predict(X_te))
        rows.append({"Model": name,
                     "CV RMSE": -cv_res["test_rmse"].mean(), "CV R2": cv_res["test_r2"].mean(),
                     "Train R2": tr["R2"],
                     "Test MAE": te["MAE"], "Test RMSE": te["RMSE"], "Test R2": te["R2"]})
        fitted[name] = model
    res = pd.DataFrame(rows).set_index("Model")
    candidates_only = res.drop(index="Mean baseline (Dummy)")
    selected = candidates_only["CV RMSE"].idxmin()          # chosen on TRAINING cross-validation only
    print(f"[{label}] best hyperparameters found:")
    for k, v in best_params.items():
        print("   ", k, "->", v)
    print(f"[{label}] Selected model (lowest CV RMSE on training data): {selected}")
    return res, fitted, selected, best_params

res1, fitted1, sel1, params1 = tune_and_select(
    lambda: make_prep_d1(BEST_FS), X_train, y_train, X_test, y_test, GRIDS, "Experiment A / Dataset 1")

## 12. Model Evaluation and Comparison  (KDD stage 12)

**Metrics in simple words**

| Metric | Meaning | Better is |
|---|---|---|
| **MAE** (Mean Absolute Error) | the average size of the mistake, in the same unit as the target (USD here) | lower |
| **RMSE** (Root Mean Squared Error) | like MAE but squares errors first, so **big mistakes are punished more**; same unit as the target | lower |
| **R-squared** | share of the variation in charges that the model explains; 1 = perfect, 0 = no better than predicting the average, can be negative | higher |

**CV columns** come from cross-validation on the training set (used to pick the model). **Test columns** come from the held-out 20 % (the honest internal evaluation). A large gap between **Train R2** and **Test R2** is a warning sign of overfitting.

In [ ]:
table1 = res1.sort_values("Test RMSE").copy()
print("Experiment A - Dataset 1 (target: charges, USD) - INTERNAL TEST evaluation")
display(table1.round(4))

In [ ]:
plot_df = res1.drop(index="Mean baseline (Dummy)").sort_values("Test RMSE")
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
for ax, metric, ttl, lab in [(axes[0], "Test MAE", "Test MAE by Model (lower is better)", "MAE (USD)"),
                             (axes[1], "Test RMSE", "Test RMSE by Model (lower is better)", "RMSE (USD)"),
                             (axes[2], "Test R2", "Test R-squared by Model (higher is better)", "R-squared")]:
    sns.barplot(x=plot_df.index, y=plot_df[metric], ax=ax, palette="viridis")
    ax.set_title(ttl); ax.set_xlabel("Model"); ax.set_ylabel(lab)
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

print("Reference: mean-baseline Test R2 =", round(res1.loc['Mean baseline (Dummy)', 'Test R2'], 4),
      "| mean-baseline Test RMSE =", round(res1.loc['Mean baseline (Dummy)', 'Test RMSE'], 1))

**How to read this:** (1) every real model should beat the dummy baseline; (2) compare *test* ranking with *CV* ranking - if they roughly agree, the selection is trustworthy; (3) note which model is selected by CV (printed above) - do not simply pick the highest test score, because that would use the test set for selection.

### 12.1 Diagnostics for the selected model: actual vs predicted, residuals, overfitting check

In [ ]:
def diagnostics(model, X_te, y_te, title, unit="USD"):
    pred = model.predict(X_te)
    resid = y_te - pred
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))

    axes[0].scatter(y_te, pred, alpha=0.6, color="steelblue")
    lims = [min(y_te.min(), pred.min()), max(y_te.max(), pred.max())]
    axes[0].plot(lims, lims, "r--", label="perfect prediction")
    axes[0].set_title(f"Actual vs Predicted - {title}")
    axes[0].set_xlabel(f"Actual ({unit})"); axes[0].set_ylabel(f"Predicted ({unit})"); axes[0].legend()

    axes[1].scatter(pred, resid, alpha=0.6, color="darkorange")
    axes[1].axhline(0, color="red", linestyle="--")
    axes[1].set_title("Residuals vs Predicted"); axes[1].set_xlabel(f"Predicted ({unit})")
    axes[1].set_ylabel(f"Residual = actual - predicted ({unit})")

    sns.histplot(resid, bins=30, kde=True, ax=axes[2], color="seagreen")
    axes[2].set_title("Distribution of Residuals"); axes[2].set_xlabel(f"Residual ({unit})"); axes[2].set_ylabel("Count")
    plt.tight_layout(); plt.show()
    return pred, resid

pred1, resid1 = diagnostics(fitted1[sel1], X_test, y_test, sel1)

r = res1.loc[sel1]
print(f"Selected model: {sel1}")
print(f"Train R2 = {r['Train R2']:.4f} | CV R2 = {r['CV R2']:.4f} | Test R2 = {r['Test R2']:.4f}")
print(f"Gap (Train R2 - Test R2) = {r['Train R2'] - r['Test R2']:.4f}")
print(f"Test MAE = {r['Test MAE']:,.0f} USD | Test RMSE = {r['Test RMSE']:,.0f} USD | mean of test charges = {y_test.mean():,.0f} USD")
print(f"Mean residual = {resid1.mean():,.0f} | residual std = {resid1.std():,.0f}")

**How to read this:** points close to the red dashed line mean good predictions. In the residual plot look for a *pattern*: a funnel shape (errors growing with predicted value) or a curve means the model is missing something. A small Train-Test R2 gap means little overfitting; a large gap means the model memorised the training data.

## 13. Interpretation: Feature Importance  (KDD stage 13)

We use **permutation importance** on the **test set**: shuffle one original feature at a time and measure how much R-squared drops. A bigger drop means the model relied on that feature more. It works for any model and is computed on the original columns (so `smoker` is one bar, not several one-hot pieces).

> **Association is not causation.** A feature being important for *prediction* does not mean that changing it would *cause* a change in cost. Nothing here is medical or financial advice.

In [ ]:
def plot_perm_importance(model, X_te, y_te, cols, title, n_repeats=20):
    pi = permutation_importance(model, X_te[cols], y_te, n_repeats=n_repeats,
                                random_state=RANDOM_STATE, scoring="r2")
    imp = pd.DataFrame({"feature": cols, "importance": pi.importances_mean, "std": pi.importances_std})
    imp = imp.sort_values("importance", ascending=True)
    plt.figure(figsize=(8, 0.5 * len(cols) + 2))
    plt.barh(imp["feature"], imp["importance"], xerr=imp["std"], color="slateblue")
    plt.title(title); plt.xlabel("Drop in R-squared when the feature is shuffled"); plt.ylabel("Feature")
    plt.tight_layout(); plt.show()
    return imp.sort_values("importance", ascending=False).reset_index(drop=True)

cols1 = ["age", "bmi", "children", "sex", "smoker", "region"] + (["obese", "smoker_obese"] if BEST_FS == "Engineered" else [])
imp1 = plot_perm_importance(fitted1[sel1], X_test, y_test, cols1,
                            f"Permutation Importance - {sel1} (Dataset 1, test set)")
display(imp1.round(4))

**How to read this:** rank the features by the printed importance. Features with importance near zero add almost nothing to this model's predictions on this dataset. If engineered features are present, remember they share information with `smoker` and `bmi`, so shuffling one of them can partly be compensated by the others - importance is shared among related features.

## 14. Two-Dataset Evaluation

### 14.1 Why there is no external validation of the Dataset 1 model
In Section 4.1 we found that the targets differ (billed charges in USD vs. yearly premium in INR) and the datasets share essentially only `age`. The Dataset 1 model needs `smoker`, `region`, `children` and `sex`, none of which exist in Dataset 2, and its output is in dollars of billed cost. Applying it to Dataset 2 would require inventing missing columns or mapping unlike targets - which would be fabrication. So, as the project brief requires for incompatible data, we run a **separate experiment** on Dataset 2.

| | Experiment A (done above) | Experiment B (below) |
|---|---|---|
| Dataset | Dataset 1 (`insurance.csv`) | Dataset 2 (`Medicalpremium.csv`) |
| Target | `charges` (USD) | `PremiumPrice` (INR) |
| Evaluation type | **Internal** train/test split | **Internal** train/test split on a *different* dataset |
| Relationship | model development | independent repeat of the same method |

> Neither experiment is *external validation* of the other. Experiment B tells us whether the **method** (same models, same metrics, same selection rule) behaves similarly on a different kind of insurance data.

### 14.2 Quick EDA for Dataset 2

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
sns.histplot(df2["PremiumPrice"], bins=30, kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Distribution of Premium Price (Dataset 2)")
axes[0].set_xlabel("Premium price (INR)"); axes[0].set_ylabel("Number of people")

sns.scatterplot(data=df2, x="Age", y="PremiumPrice", alpha=0.5, ax=axes[1], color="darkorange")
axes[1].set_title("Premium Price vs Age"); axes[1].set_xlabel("Age (years)"); axes[1].set_ylabel("Premium price (INR)")

sns.scatterplot(data=df2, x="BMI", y="PremiumPrice", alpha=0.5, ax=axes[2], color="seagreen")
axes[2].set_title("Premium Price vs BMI (derived from height and weight)")
axes[2].set_xlabel("BMI (kg/m^2)"); axes[2].set_ylabel("Premium price (INR)")
plt.tight_layout(); plt.show()

# mean premium by each yes/no health flag
flag_rows = []
for c in binary_cols_d2 + ["NumberOfMajorSurgeries"]:
    for v, grp in df2.groupby(c)["PremiumPrice"]:
        flag_rows.append({"feature": c, "value": v, "count": len(grp), "mean premium": grp.mean()})
flags = pd.DataFrame(flag_rows)

plt.figure(figsize=(9, 5))
plot_flags = flags[flags["feature"].isin(binary_cols_d2)]
sns.barplot(data=plot_flags, x="mean premium", y="feature", hue="value", palette="Set2")
plt.title("Average Premium by Health Flag (0 = no, 1 = yes)")
plt.xlabel("Average premium (INR)"); plt.ylabel("Health flag"); plt.legend(title="Flag value")
plt.tight_layout(); plt.show()

corr2 = df2[["Age", "BMI", "NumberOfMajorSurgeries"] + binary_cols_d2 + ["PremiumPrice"]].corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr2, annot=True, fmt=".2f", cmap="coolwarm", center=0, annot_kws={"size": 7})
plt.title("Correlation Heatmap (Dataset 2)")
plt.tight_layout(); plt.show()

print("Correlation with PremiumPrice (sorted):")
print(corr2["PremiumPrice"].drop("PremiumPrice").sort_values(ascending=False).round(3))
print("\nNumber of distinct premium values:", df2["PremiumPrice"].nunique())
print(df2.groupby("NumberOfMajorSurgeries")["PremiumPrice"].agg(["count", "mean"]).round(0))

**How to read this:** compare the bars for flag = 0 vs flag = 1 and the sorted correlations to see which attributes move together with the premium. Check the printed number of distinct premium values: if it is small, the target is "tiered" rather than a continuous billed amount, which affects how the models behave. These are associations in the sample only.

### 14.3 Experiment B - modelling Dataset 2
Same methodology as Experiment A: 80/20 split (not stratified because there is no obvious grouping variable), pipeline with scaling for numeric columns, 0/1 flags passed through unchanged, five models plus the dummy baseline, grid search on the tree models, selection by training-set cross-validation, final report on the held-out test set.

**Feature choice:** `Height` and `Weight` are replaced by the derived `BMI` to avoid duplicated information (a design decision you can mention in your presentation; you can test the alternative yourself).

In [ ]:
FEATURES_D2_NUM = ["Age", "BMI", "NumberOfMajorSurgeries"]
FEATURES_D2_BIN = binary_cols_d2
FEATURES_D2 = FEATURES_D2_NUM + FEATURES_D2_BIN

X2 = df2[FEATURES_D2]
y2 = df2["PremiumPrice"]
assert "PremiumPrice" not in X2.columns, "Target leaked into features!"

X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=RANDOM_STATE)
print("Dataset 2 -> training rows:", len(X2_train), "| test rows:", len(X2_test))

def make_prep_d2():
    return ColumnTransformer([("num", StandardScaler(), FEATURES_D2_NUM),
                              ("bin", "passthrough", FEATURES_D2_BIN)], remainder="drop")

res2, fitted2, sel2, params2 = tune_and_select(
    make_prep_d2, X2_train, y2_train, X2_test, y2_test, GRIDS, "Experiment B / Dataset 2")

print("\nExperiment B - Dataset 2 (target: PremiumPrice, INR) - INTERNAL TEST evaluation")
display(res2.sort_values("Test RMSE").round(4))

In [ ]:
plot_df2 = res2.drop(index="Mean baseline (Dummy)").sort_values("Test RMSE")
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
for ax, metric, ttl, lab in [(axes[0], "Test MAE", "Test MAE by Model - Dataset 2 (lower is better)", "MAE (INR)"),
                             (axes[1], "Test RMSE", "Test RMSE by Model - Dataset 2 (lower is better)", "RMSE (INR)"),
                             (axes[2], "Test R2", "Test R-squared by Model - Dataset 2 (higher is better)", "R-squared")]:
    sns.barplot(x=plot_df2.index, y=plot_df2[metric], ax=ax, palette="magma")
    ax.set_title(ttl); ax.set_xlabel("Model"); ax.set_ylabel(lab); ax.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

pred2, resid2 = diagnostics(fitted2[sel2], X2_test, y2_test, f"{sel2} (Dataset 2)", unit="INR")
r2row = res2.loc[sel2]
print(f"Selected model: {sel2}")
print(f"Train R2 = {r2row['Train R2']:.4f} | CV R2 = {r2row['CV R2']:.4f} | Test R2 = {r2row['Test R2']:.4f} | gap = {r2row['Train R2'] - r2row['Test R2']:.4f}")
print(f"Test MAE = {r2row['Test MAE']:,.0f} INR | Test RMSE = {r2row['Test RMSE']:,.0f} INR | mean test premium = {y2_test.mean():,.0f} INR")

imp2 = plot_perm_importance(fitted2[sel2], X2_test, y2_test, FEATURES_D2,
                            f"Permutation Importance - {sel2} (Dataset 2, test set)")
display(imp2.round(4))

**How to read this:** use the same checklist as Experiment A - beat the baseline? small Train-Test gap? residual pattern? Then read which features the model relies on. The permutation importance describes **this model on this dataset**, not insurers in general.

### 14.4 Cautious comparison of the two experiments
Raw errors cannot be compared (USD vs INR, different scales). We compare only **unit-free** quantities: R-squared, and error divided by the average test target ("relative error"). Even these are only indicative, because the datasets, features, populations and sample sizes differ.

In [ ]:
def summary_row(label, n_rows, target, res, sel, y_te, imp):
    r = res.loc[sel]
    return {"Experiment": label, "Rows (after dedup)": n_rows, "Target": target, "Selected model": sel,
            "Test R2": r["Test R2"],
            "MAE / mean target": r["Test MAE"] / y_te.mean(),
            "RMSE / mean target": r["Test RMSE"] / y_te.mean(),
            "Train-Test R2 gap": r["Train R2"] - r["Test R2"],
            "Top-3 features": ", ".join(imp["feature"].head(3))}

compare = pd.DataFrame([
    summary_row("A: Dataset 1", len(df1), "charges (USD)", res1, sel1, y_test, imp1),
    summary_row("B: Dataset 2", len(df2), "PremiumPrice (INR)", res2, sel2, y2_test, imp2),
]).set_index("Experiment")
display(compare.T)

# Which model family won in each experiment (CV ranking, tuned models only)
rank = pd.DataFrame({
    "A: CV RMSE rank": res1.drop(index="Mean baseline (Dummy)")["CV RMSE"].rank(),
    "B: CV RMSE rank": res2.drop(index="Mean baseline (Dummy)")["CV RMSE"].rank()})
print("Model ranking by cross-validated RMSE (1 = best):")
display(rank.sort_values("A: CV RMSE rank"))

**Limitations of this comparison (always state these in your presentation):**
* **Different targets:** billed medical cost (USD) vs. insurance premium (INR). A premium is set by an insurer's pricing rules; a billed cost is what care costs.
* **Different features:** the two datasets describe people with different variables, so the models answer different questions.
* **Different populations and sources:** Dataset 1 is US-based teaching data (documentation describes it as simulated from census statistics in some sources); Dataset 2's population is not documented in the file.
* **Different sample sizes** (see the table) affect how stable the scores are.
* Relative error and R-squared are indicative, **not** a proof that one dataset is "easier".

## 15. Final Conclusions and Limitations  (KDD stage 14)

The cell below assembles a factual summary **from your own run**, so your conclusions cannot drift away from your results.

In [ ]:
r1 = res1.loc[sel1]
r2_ = res2.loc[sel2]
print("=" * 78)
print("RESULT SUMMARY (generated from this run)")
print("=" * 78)
print(f"Experiment A - Dataset 1 ({len(df1)} rows after removing {n_dup1} duplicate row(s)), target: charges (USD)")
print(f"  Feature set used     : {BEST_FS}")
print(f"  Selected model       : {sel1}  (lowest cross-validated RMSE on training data)")
print(f"  Internal test result : MAE {r1['Test MAE']:,.0f} | RMSE {r1['Test RMSE']:,.0f} | R2 {r1['Test R2']:.3f}")
print(f"  Mean-baseline R2     : {res1.loc['Mean baseline (Dummy)', 'Test R2']:.3f}")
print(f"  Train-Test R2 gap    : {r1['Train R2'] - r1['Test R2']:.3f}")
print(f"  Most important features (permutation): {', '.join(imp1['feature'].head(3))}")
print()
print(f"Experiment B - Dataset 2 ({len(df2)} rows after removing {n_dup2} duplicate row(s)), target: PremiumPrice (INR)")
print(f"  Selected model       : {sel2}")
print(f"  Internal test result : MAE {r2_['Test MAE']:,.0f} | RMSE {r2_['Test RMSE']:,.0f} | R2 {r2_['Test R2']:.3f}")
print(f"  Mean-baseline R2     : {res2.loc['Mean baseline (Dummy)', 'Test R2']:.3f}")
print(f"  Train-Test R2 gap    : {r2_['Train R2'] - r2_['Test R2']:.3f}")
print(f"  Most important features (permutation): {', '.join(imp2['feature'].head(3))}")
print()
print("External validation of the Dataset 1 model on Dataset 2: NOT performed (incompatible target and features - see Section 4.1).")

### What you can conclude (fill in using the printed summary)
1. Which model did cross-validation select for each dataset, and did it clearly beat the mean baseline?
2. Which features were most associated with predicted cost in each dataset? State these as **associations**.
3. Did either experiment show a large Train-Test gap (overfitting)?
4. Did the same model family win in both experiments, or different ones? (see the rank table in 14.4)

### Limitations
* **Dataset 1** is a small (about one and a third thousand rows), widely reused teaching dataset; its documentation in some sources describes it as simulated. Results may not represent real insurer portfolios.
* **No external validation** was possible: the second dataset has a different target concept, currency and feature set.
* A **single** train/test split was used for the final test score; the test score varies with the split. Cross-validation scores give a second view.
* Models predict **associations**, not causes. Permutation importance is shared among correlated features.
* Charges/premiums depend on things not recorded here (medical history, claims history, plan type, income, year, insurer rules).
* The datasets contain personal and health-related attributes; using such models for real pricing raises **fairness and privacy** issues (e.g. using sex or region). This notebook is for learning, not for pricing decisions.

### Future improvements
* Tune a wider grid or use randomised search; try `RidgeCV`/`ElasticNet`, XGBoost/LightGBM.
* Model `log(charges)` and compare; add repeated cross-validation and confidence intervals for the test metrics.
* Add SHAP values for richer explanations.
* Find a second dataset with the **same target definition** (billed cost in the same currency and similar features) to allow genuine external validation.
* Check fairness across groups (sex, region) before any real-world use.

## 16. Quality Checklist (run this last)
This cell re-checks the key safeguards programmatically.

In [ ]:
checks = {
    "Datasets are different files with different targets": df1_raw.columns.tolist() != df2_raw.columns.tolist(),
    "Only age-like column is shared (so no row-level duplication across datasets)": shared == ["age"],
    "No target column in Dataset 1 features": "charges" not in X_train.columns,
    "No target column in Dataset 2 features": "PremiumPrice" not in X2_train.columns,
    "Train and test sets do not overlap (Dataset 1)": len(set(X_train.index) & set(X_test.index)) == 0,
    "Train and test sets do not overlap (Dataset 2)": len(set(X2_train.index) & set(X2_test.index)) == 0,
    "Same split used for all Dataset 1 models": all(m.predict(X_test).shape[0] == len(y_test) for m in fitted1.values()),
    "Dataset 1 model was selected by lowest CV RMSE (training data)": sel1 == res1.drop(index="Mean baseline (Dummy)")["CV RMSE"].idxmin(),
    "Dataset 2 model was selected by lowest CV RMSE (training data)": sel2 == res2.drop(index="Mean baseline (Dummy)")["CV RMSE"].idxmin(),
}
for k, v in checks.items():
    print(("PASS  " if v else "FAIL  ") + k)
if not checks["Only age-like column is shared (so no row-level duplication across datasets)"]:
    print("\nNote: shared columns were:", shared, "- inspect these before claiming the datasets are unrelated.")

## 17. Download all charts
Every chart shown above was also saved as a PNG in the `figures` folder. This cell zips them and starts a download of `figures.zip` (in Colab). Unzip it and copy the images into the `images/` folder of your GitHub repository.

In [ ]:
import shutil
shutil.make_archive("figures", "zip", "figures")
print("Saved", len(os.listdir("figures")), "charts:")
for f in sorted(os.listdir("figures")):
    print("  ", f)
try:
    from google.colab import files
    files.download("figures.zip")          # browser download in Colab
except ImportError:
    print("Not running in Colab: find the charts in the 'figures' folder and in figures.zip")